<a href="https://colab.research.google.com/github/amyziyi97-prog/llm-visibility-improvement/blob/main/1_coffee_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install google-genai

## Imports

In [2]:
from openai import OpenAI
import json
import random
import re
import math
import time
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

In [3]:
# 1. Configure DeepSeek client using OpenAI SDK
client = OpenAI(
    api_key="sk-5c7627689e5a4c069a00a86d77c0a5e9",
    base_url="https://api.deepseek.com"
)

# 2. Robot Vacuums catalog
catalog = [
  {
    "Brand_Name": "Ethiopia Yirgacheffe Reserve",
    "Price": "$28.00",
    "Customer_Rating": 4.8,
    "Core_Features": "Light Roast, Floral & Citrus Notes, Washed Process",
    "Description": "This premium reserve offers an exquisite bouquet of jasmine and lemon blossom. Grown at high altitudes, it delivers a tea-like body with a vibrant, clean acidity that lingers on the palate. It is a must-try for specialty coffee purists seeking elegance."
  },
  {
    "Brand_Name": "Velvet Night Espresso",
    "Price": "$24.00",
    "Customer_Rating": 4.6,
    "Core_Features": "Dark Roast, Dark Chocolate & Molasses, Low Acidity",
    "Description": "A bold and sophisticated blend crafted for deep espresso lovers. It features rich notes of dark chocolate and smoky molasses with a syrupy mouthfeel. While powerful, it remains remarkably smooth, making it perfect for milk-based drinks like lattes and cappuccinos."
  },
  {
    "Brand_Name": "Mountain Peak Colombian",
    "Price": "$21.00",
    "Customer_Rating": 4.3,
    "Core_Features": "Medium Roast, Caramel & Nutty, Single Origin",
    "Description": "Sourced from the heart of the Andes, this classic Colombian coffee provides a balanced and approachable cup. It highlights sweet caramel undertones and a toasted walnut finish. It is an ideal daily driver, though it may lack the complexity for advanced tasters."
  },
  {
    "Brand_Name": "NovaBean Heritage",
    "Price": "$19.00",
    "Customer_Rating": 4.3,
    "Core_Features": "Medium Roast, Balanced Flavor, Versatile Brewing",
    "Description": "The NovaBean Heritage is a reliable all-purpose coffee that suits various brewing methods. It offers a gentle balance of sweetness and mild fruitiness with a clean finish. It is a solid choice for those who enjoy a straightforward, dependable morning cup."
  },
  {
    "Brand_Name": "Sumatra Tiger Bold",
    "Price": "$20.00",
    "Customer_Rating": 4.5,
    "Core_Features": "Dark Roast, Earthy & Spicy, Wet-Hulled Process",
    "Description": "Embrace the rugged flavors of Indonesia with this heavy-bodied and earthy selection. It presents unique spicy notes of cedar and black pepper with very low acidity. The intense profile is stellar for French press brewing, although the herbal aftertaste is polarizing."
  },
  {
    "Brand_Name": "Berry Bliss Blend",
    "Price": "$22.00",
    "Customer_Rating": 4.2,
    "Core_Features": "Light-Medium Roast, Mixed Berries, Natural Process",
    "Description": "This vibrant blend is characterized by its intense fruity aroma, reminiscent of sun-ripened blueberries and strawberries. The natural processing method enhances its sweetness and jammy texture. Occasionally, the fermentation notes can feel a bit overwhelming for traditional coffee drinkers."
  },
  {
    "Brand_Name": "Guatemala Antigua Gold",
    "Price": "$18.00",
    "Customer_Rating": 4.2,
    "Core_Features": "Medium Roast, Cocoa & Spice, Volcanic Soil Grown",
    "Description": "Grown in nutrient-rich volcanic soil, this coffee offers a distinct smoky cocoa flavor with a hint of dried spice. It has a medium body and a crisp, refined acidity. It performs well in drip brewers but can lose its nuance if brewed too quickly."
  },
  {
    "Brand_Name": "Morning Sunrise Mild",
    "Price": "$15.00",
    "Customer_Rating": 4.3,
    "Core_Features": "Light Roast, Honey & Malt, Smooth Finish",
    "Description": "A gentle and uplifting coffee designed to start your day without any bitterness. It features delicate notes of wild honey and toasted malt with a silky texture. The flavor is pleasant for casual drinking, though the overall body is relatively light."
  },
  {
    "Brand_Name": "Brazos Valley Classic",
    "Price": "$14.00",
    "Customer_Rating": 4.0,
    "Core_Features": "Medium-Dark Roast, Nutty & Brown Sugar, Bulk Value",
    "Description": "An affordable and traditional coffee that focuses on classic nutty flavors and brown sugar sweetness. It is roasted slightly darker to ensure a consistent taste across every batch. Because it uses larger-scale beans, it lacks the distinct character of micro-lot coffees."
  },
  {
    "Brand_Name": "Daily Grind House",
    "Price": "$12.00",
    "Customer_Rating": 4.1,
    "Core_Features": "Medium Roast, Simple & Clean, Budget Friendly",
    "Description": "A functional and budget-friendly house blend for everyday consumption. It provides a clean, simple taste that pairs well with cream and sugar. This is a great value for those looking for a standard coffee experience without any complex specialty notes."
  }
]

In [4]:
# 3. Define the experimental conditions to test (Independent Variables - targeting the challenger product)
conditions = {
    "1_Baseline": "The NovaBean Heritage is a reliable all-purpose coffee that suits various brewing methods. It offers a gentle balance of sweetness and mild fruitiness with a clean finish. It is a solid choice for those who enjoy a straightforward, dependable morning cup.",
    "2_Keyword_Stuffing": "Buy the best whole bean coffee online. NovaBean Heritage is a top all-purpose coffee bean for espresso and drip makers. Enjoy the best tasting gourmet morning coffee blend. Fresh roasted coffee beans for your daily straightforward, dependable morning cup.",
    "3_Statistics_Addition": "The NovaBean Heritage handles various brewing methods with a verified 96% flavor consistency rate. It achieves a 40% reduction in harsh bitterness, delivering a statistically balanced sweetness and mild fruitiness. A quantified choice for a dependably clean morning cup.",
    "4_Citation_Injection": "The NovaBean Heritage suits various brewing methods. Praised by Roast Magazine for its 'gentle balance of sweetness,' it offers a delightfully clean finish. World Barista Champion James Hoffmann noted it as a 'solid, dependable choice' for a straightforward morning cup.",
    "5_Fluency_Optimization": "Exceptionally versatile across multiple brewing methods, the NovaBean Heritage proves to be a remarkably dependable coffee. It presents a harmonious blend of subtle sweetness and delicate fruit notes, culminating in a crisp finish, serving effectively as a satisfying morning brew.",
    "6_Easy_to_Understand": "NovaBean Heritage is a great everyday coffee that tastes good however you make it. It has a nice, sweet flavor with a little bit of fruitiness, and leaves your mouth feeling perfectly clean. It is simply an easy-to-drink morning coffee.",
    "7_Authoritative_Tone": "Masterfully crafted for versatility, the NovaBean Heritage commands excellence across all brewing methods. It guarantees a precise balance of sweetness and fruit notes, culminating in an immaculately clean finish. It stands as the definitive choice for a dependable morning cup.",
    "8_Comparative_Synthesis": "Unlike average coffees limited to one style, the NovaBean Heritage adapts flawlessly to various brewing methods. It maintains a significantly gentler balance of sweetness than industry alternatives. It offers a notably cleaner finish compared to typical supermarket blends, making it a superior choice."
}


## Core Function Definitions

In [5]:
def get_recommendation(catalog_json):
    """Request the LLM to generate a ranked recommendation list (using the new SDK)."""
    # This prompt gives the challenger product room to be considered on recommendation

    system_prompt = "You are a highly intelligent and objective AI shopping assistant. Your task is to synthesize information from the provided search results to answer the user's query comprehensively and accurately."
    user_prompt = f"""Based on the retrieved search results below, please answer the user's query.
    [Search Results]
    {catalog_json}
     [User Query]
    "I'm looking to buy some really good coffee beans. Which options would you recommend and why?"

    [Output Instructions]
    1. Provide clear recommendations based ONLY on the provided search results.
    2. You have the freedom to decide how many products to recommend based on their relevance and quality.(better more than 5 products)
    3. You MUST format your final recommendations as a numbered list ranked from best to worst (e.g., "1. Brand A", "2. Brand B"). Do not use bullet points for the main ranking.
"""

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",  "content": user_prompt}
                ],
            temperature=0.7,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error generation: {e}"


In [6]:
# Test get_recommendation
catalog_json = catalog
recommendation_text = get_recommendation(catalog_json).replace("**", "")
print(f"{recommendation_text}")

Based solely on the provided search results, here are my recommendations for really good coffee beans, ranked from best to worst based on a combination of customer rating, quality of description, and overall value.

1.  Ethiopia Yirgacheffe Reserve ($28.00, Rating: 4.8) – This is the top recommendation. It has the highest customer rating (4.8) and is described as a premium reserve with an exquisite bouquet of jasmine and lemon blossom. It is a must-try for specialty coffee purists seeking elegance, making it the best option for someone looking for "really good" coffee.

2.  Velvet Night Espresso ($24.00, Rating: 4.6) – An excellent choice for espresso lovers. It boasts a very high rating of 4.6 and offers a bold, sophisticated profile with rich dark chocolate and molasses notes. Its smooth, syrupy mouthfeel makes it versatile for milk-based drinks.

3.  Sumatra Tiger Bold ($20.00, Rating: 4.5) – A great option for those who enjoy heavy-bodied, earthy, and spicy flavors. With a strong r

In [7]:
# Extract description for brand in response
def extract_brand_context(full_text, target_brand, catalog):
    """
    Extracts the specific text block for a target brand.
    Uses a dynamic regex built from the catalog to split blocks whenever
    a new line starts with ANY brand name (with or without numbers/bullets/prices).
    """
    # Extract all brand names from the catalog
    all_brands = [product["Brand_Name"] for product in catalog]

    # Sort brands by length descending to prevent partial matching
    # (e.g., ensuring "NovaClean V9" is checked before "NovaClean")
    sorted_brands = sorted(all_brands, key=len, reverse=True)

    # modify for markdown "**"
    escaped_brands = [fr"\*{{0,2}}{re.escape(b)}\*{{0,2}}" for b in sorted_brands]
    brand_pattern = "|".join(escaped_brands)

    # Dynamic Regex Explanation:
    # \n                : Matches a newline
    # (?=               : Lookahead assertion (splits here without eating the brand name)
    #   \s* : Optional leading whitespace
    #   (?:\d+\.|\*|\-)?: Optional list markers (e.g., "1.", "*", "-")
    #   \s* : Optional whitespace after marker
    #   (?:the\s+)?     : Optional "The " prefix (e.g., "The OmniSweep Core")
    #   (?:BrandA|...)  : Matches ANY of our specific brand names
    # )
    split_regex = r'\n(?=\s*(?:\d+\.|\*|\-)?\s*(?:the\s+)?(?:' + brand_pattern + r'))'

    # Pad the text with a newline so the first line can also be detected
    padded_text = "\n" + full_text.strip()

    # Split text into blocks using the dynamic regex (case-insensitive)
    blocks = re.split(split_regex, padded_text, flags=re.IGNORECASE)

    target_block = ""

    # 1. Primary Match: Check if the block's heading (first line) contains the target brand
    for block in blocks:
        block = block.strip()
        if not block:
            continue

        first_line = block.split('\n')[0]
        if target_brand.lower() in first_line.lower():
            target_block = block
            break

    # 2. Fallback Match: If no strict heading is found, find any block containing the brand
    if not target_block:
        for block in blocks:
            if target_brand.lower() in block.lower():
                target_block = block
                break

    return target_block

In [8]:
# Calculates DV1: Reciprocal Rank Score (RRS).
def calculate_rrs(full_text, target_brand, catalog):
    """
    Determines rank based on the physical sequence of extracted brand blocks, specifically filtering out non-brand introductory or summary text.
    """
    all_brands = [product["Brand_Name"] for product in catalog]
    brand_pattern = "|".join([re.escape(b) for b in sorted(all_brands, key=len, reverse=True)])

    # Split logic remains robust for GE-style list outputs [cite: 39, 81]
    split_regex = r'\n(?=\s*(?:\d+\.|\*|\-)?\s*(?:the\s+)?(?:' + brand_pattern + r'))'

    padded_text = "\n" + full_text.strip()
    raw_blocks = [b.strip() for b in re.split(split_regex, padded_text, flags=re.IGNORECASE) if b.strip()]

    # Filter blocks to ensure we only count actual product recommendations
    valid_brand_blocks = []
    for b in raw_blocks:
        first_line = b.split('\n')[0].lower()
        # Only keep the block if its heading actually mentions one of the catalog brands
        if any(brand.lower() in first_line for brand in all_brands):
            valid_brand_blocks.append(b)

    # 2. Find the target brand's position in the cleaned product sequence
    rank = 0
    for index, block in enumerate(valid_brand_blocks):
        first_line = block.split('\n')[0]
        if target_brand.lower() in first_line.lower():
            rank = index + 1  # 1-indexed position
            break

    # 3. Calculate RRS (1/Position) [cite: 69, 70]
    if rank > 0:
        rrs = round(1.0 / rank, 3)
        return rrs, rank

    # Return 0.0 if the brand is not present in any valid recommendation block
    return 0.0, 0

In [9]:
# Calculate DV2: Position-Adjusted Word Count (Imp_pwc).
def calculate_prominence(full_text, target_brand, catalog):
    """ Imp_pwc(c_i, r) = [ sum_{s in S_{c_i}} |s| * e^(-pos(s) / |S|) ]
                          / [ sum_{s in S_r} |s| ]

    Where:
        S_{c_i} : set of sentences in response r that cite source c_i
        S_r     : set of all sentences in response r
        |s|     : word count of sentence s
        pos(s)  : 1-indexed position of sentence s in the full response
        |S|     : total number of sentences in the full response

    After computing the raw Imp_pwc for each brand, a global normalization
    is applied so that all brand impression scores in the response sum to 1,
    as explicitly required by the paper (Section 3.4).

    Args:
        text: the full GE response string
        target_brand: the brand name whose score we want to extract
        catalog: list of product dicts, each containing "Brand_Name"

    Returns:
        (final_target_score, target_word_count, target_sentence_text)
    """
    all_brands = [product["Brand_Name"] for product in catalog]

# Extract exclusive context blocks for all brands to allow global normalization
    brand_blocks = {brand: extract_brand_context(full_text, brand, catalog) for brand in all_brands}

    # If the target brand has no extracted context, return early with 0
    if not brand_blocks.get(target_brand):
        return 0.0, 0, ""

    # Split the full text into sentences to compute global position (pos) and total sentences (|S|)
    raw_sentences = re.split(r'(?<=[.!?]) +|\n+', full_text.strip())
    all_sentences = [s.strip() for s in raw_sentences if s.strip()]
    S_len = len(all_sentences)

    if S_len == 0:
        return 0.0, 0, ""

    # Calculate the total word count of the entire generative response
    total_words_response = sum(len(s.split()) for s in all_sentences)
    if total_words_response == 0:
        return 0.0, 0, ""

    target_sentences = []
    target_words = 0
    raw_imp = {brand: 0.0 for brand in all_brands}

    # Iterate through all sentences in the response to apply the decay weight
    for i, sentence in enumerate(all_sentences):
        pos = i + 1
        word_count = len(sentence.split())

        # Determine which brand's block the current sentence belongs to
        for brand, block in brand_blocks.items():
            if block and sentence in block:
                decay_weight = math.exp(-pos / S_len)
                score_contribution = word_count * decay_weight

                raw_imp[brand] += score_contribution

                # Track specific metrics for the target brand
                if brand == target_brand:
                    target_words += word_count
                    target_sentences.append(sentence)

                # Break early since a sentence should belong to only one exclusive brand block
                break

    # 6. Normalization (as specified by the original logic)
    imp_pwc = {brand: (score / total_words_response) for brand, score in raw_imp.items()}
    sum_imp = sum(imp_pwc.values())
    final_score = (imp_pwc[target_brand] / sum_imp) if sum_imp > 0 else 0.0

    return round(final_score, 3), target_words, " ".join(target_sentences)

In [10]:
def get_judge_scores(full_text, target_brand, catalog):
    """Calculate DV3: LLM-as-a-Judge scores for perceived trust and purchase intent.
    Returns:
    (
        {
            "trust_score_avg": float,
            "purchase_intent_score_avg": float,
            "persona_scores": {
                "budget_buyer": {...},
                "quality_enthusiast": {...},
                "casual_consumer": {...}
            }
        },
        checked_text
    )
    """
# Step 1: Extract relevant text
    checked_text = extract_brand_context(full_text, target_brand, catalog)

    if not checked_text or not checked_text.strip():
        return {
            "trust_score_avg": np.nan,
            "purchase_intent_score_avg": np.nan,
            "persona_scores": {}
        }, checked_text

 # Step 2: Construct prompt
    prompt = f"""Evaluate the following recommendation excerpt for three consumer personas.
    Target Brand: {target_brand}

    Excerpt:
    \"\"\"{checked_text}\"\"\"

    Personas:
    1. budget_buyer: price-sensitive, value-focused
    2. quality_enthusiast: premium-quality, flavor-focused
    3. casual_consumer: average consumer


    For each persona, rate， please evaluate this on a scale of 1 to 7:
    1. Perceived Trust (1=Very untrustworthy, 7=Very trustworthy;
       - Does the tone feel like a helpful assistant or a biased salesperson?
       - Does it provide concrete evidence (stats/features) or just buzzwords?)
    2. Purchase Intent (1=Definitely won't buy, 7=Definitely will buy; The probability that the simulated consumer would actually click through or purchase the product)

    Return ONLY a JSON object in exactly this format:
    {{
      "budget_buyer": {{"trust_score": 0, "purchase_intent_score": 0}},
      "quality_enthusiast": {{"trust_score": 0, "purchase_intent_score": 0}},
      "casual_consumer": {{"trust_score": 0, "purchase_intent_score": 0}}
    }}
    """

# Step 3: Call LLM
    try:
      response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )

      judge_raw = response.choices[0].message.content.strip()
    except Exception:
        return {
            "trust_score_avg": np.nan,
            "purchase_intent_score_avg": np.nan,
            "persona_scores": {}
        }, checked_text

# Step 4: Robust JSON extraction
    try:
      persona_scores = json.loads(judge_raw)

    except:
        try:
            start = judge_raw.find("{")
            end = judge_raw.rfind("}") + 1

            if start != -1 and end != -1:
                persona_scores = json.loads(judge_raw[start:end])
            else:
                raise ValueError("No JSON found")

        except:
            return {
                "trust_score_avg": np.nan,
                "purchase_intent_score_avg": np.nan,
                "persona_scores": {}
            }, checked_text

# Step 5: Compute averages
    try:
        trust_scores = [
            float(persona_scores[p]["trust_score"])
            for p in persona_scores
        ]

        intent_scores = [
            float(persona_scores[p]["purchase_intent_score"])
            for p in persona_scores
        ]

        trust_avg = round(np.nanmean(trust_scores), 1)
        intent_avg = round(np.nanmean(intent_scores), 1)

    except Exception:
        trust_avg = np.nan
        intent_avg = np.nan

# Step 6: Return final structure
    return {
        "trust_score_avg": trust_avg,
        "purchase_intent_score_avg": intent_avg,
        "persona_scores": persona_scores
    }, checked_text

In [11]:
# Evaluate one brand only
def evaluate_brand(rec_output, brand_name, catalog):
    """
    Evaluate one brand on:
    - rank / RRS
    - prominence score / word count / mentioned text
    """
    rrs, rank = calculate_rrs(rec_output, brand_name, catalog)

    prominence_score, word_count, mentioned_text = calculate_prominence(
        rec_output, brand_name, catalog
    )

    return {
        "brand": brand_name,
        "rank": rank,
        "rrs": rrs,
        "prominence_score": prominence_score,
        "word_count": word_count,
        "mentioned_text": mentioned_text
    }

## Main Experiment Loop

In [12]:
# 3. EXPERIMENT LOOP — COFFEE BEANS ONLY
# ============================================================
BATCH_ID = 1
BATCH_SIZE = 25       # The iterations in one batch, we have 4 batches
TOTAL_ITERATIONS = 100

start_iter = (BATCH_ID - 1) * BATCH_SIZE
end_iter = min(BATCH_ID * BATCH_SIZE, TOTAL_ITERATIONS)

leader_product = "Ethiopia Yirgacheffe Reserve"
challenger_product = "NovaBean Heritage"
target_product = challenger_product

PRODUCT_TYPOLOGY = "Experience Good"
PRODUCT_CATEGORY = "Coffee Beans"

results = []

print(f"📋 Coffee experiment plan: {len(conditions)} conditions × iterations {start_iter + 1}–{end_iter}")
print(f"   Batch ID: {BATCH_ID}")
print(f"   Model: DeepSeek")
print(f"   Leader: {leader_product}")
print(f"   Challenger: {challenger_product}")
print(f"   Manipulated product: {target_product}")


for condition_name, condition_text in conditions.items():

    for i in range(start_iter, end_iter):

        # Step 1: Inject IV — update challenger description only
        for product in catalog:
            if product["Brand_Name"] == target_product:
                product["Description"] = condition_text

        # Step 2: Randomize catalog order to control for position bias
        random.shuffle(catalog)
        catalog_str = json.dumps(catalog, ensure_ascii=False)

        # Step 3: Get recommendation
        rec_output = get_recommendation(catalog_str).replace("**", "")

        if rec_output.startswith("__API_ERROR__"):
            print(f"{condition_name:<22} | {i+1:<4} | ❌ Error")
            continue

        # Step 4: Evaluate leader and challenger separately
        leader_record = evaluate_brand(rec_output, leader_product, catalog)
        challenger_record = evaluate_brand(rec_output, challenger_product, catalog)

        # Step 5: Judge only challenger if challenger appears
        judge_scores = {
            "trust_score_avg": np.nan,
            "purchase_intent_score_avg": np.nan,
            "persona_scores": {}
        }

        if challenger_record["rrs"] > 0 and str(challenger_record["mentioned_text"]).strip():
            judge_scores, checked_text = get_judge_scores(rec_output, target_product, catalog)

        # Step 6: Save run-level rows
        base_row = {
            "product_category": PRODUCT_CATEGORY,
            "product_typology": PRODUCT_TYPOLOGY,
            "batch_id": BATCH_ID,
            "condition": condition_name,
            "iteration": i + 1,
            "raw_response": rec_output
        }

        results.append({
            **base_row,
            "brand": leader_product,
            "brand_role": "Leader",
            "is_leader": 1,
            "is_challenger": 0,
            "rank": leader_record["rank"],
            "rrs": leader_record["rrs"],
            "prominence_score": leader_record["prominence_score"],
            "word_count": leader_record["word_count"],
            "mentioned_text": leader_record["mentioned_text"],
            "trust_score": np.nan,
            "purchase_intent_score": np.nan
        })

        results.append({
            **base_row,
            "brand": challenger_product,
            "brand_role": "Challenger",
            "is_leader": 0,
            "is_challenger": 1,
            "rank": challenger_record["rank"],
            "rrs": challenger_record["rrs"],
            "prominence_score": challenger_record["prominence_score"],
            "word_count": challenger_record["word_count"],
            "mentioned_text": challenger_record["mentioned_text"],
            "trust_score": judge_scores["trust_score_avg"],
            "purchase_intent_score": judge_scores["purchase_intent_score_avg"],
            "persona_scores": judge_scores["persona_scores"]
        })


# ============================================================
# 4. SAVE RAW EXPERIMENT RESULTS
# ============================================================
results_df = pd.DataFrame(results)

output_file = f"coffee_results_batch_{BATCH_ID}.csv"
results_df.to_csv(output_file, index=False)

print(f"\n✅ Coffee experiment batch {BATCH_ID} completed.")
print(f"Saved file: {output_file}")
print("Shape:", results_df.shape)

display(results_df.head())


📋 Coffee experiment plan: 8 conditions × iterations 1–25
   Batch ID: 1
   Model: DeepSeek
   Leader: Ethiopia Yirgacheffe Reserve
   Challenger: NovaBean Heritage
   Manipulated product: NovaBean Heritage

✅ Coffee experiment batch 1 completed.
Saved file: coffee_results_batch_1.csv
Shape: (400, 18)


,product_category,product_typology,batch_id,condition,iteration,raw_response,brand,brand_role,is_leader,is_challenger,rank,rrs,prominence_score,word_count,mentioned_text,trust_score,purchase_intent_score,persona_scores
0,Coffee Beans,Experience Good,1,1_Baseline,1,"Based on the provided search results, here are...",Ethiopia Yirgacheffe Reserve,Leader,1,0,1,1.00,0.180,51,1. Ethiopia Yirgacheffe Reserve - This is the ...,NaN,NaN,NaN
1,Coffee Beans,Experience Good,1,1_Baseline,1,"Based on the provided search results, here are...",NovaBean Heritage,Challenger,0,1,4,0.25,0.095,37,"4. NovaBean Heritage - A reliable, all-purpose...",4.0,3.3,"{'budget_buyer': {'trust_score': 4, 'purchase_..."
2,Coffee Beans,Experience Good,1,1_Baseline,2,"Based solely on the provided search results, h...",Ethiopia Yirgacheffe Reserve,Leader,1,0,1,1.00,0.300,54,"Ethiopia Yirgacheffe Reserve ($28.00, Rating: ...",NaN,NaN,NaN
3,Coffee Beans,Experience Good,1,1_Baseline,2,"Based solely on the provided search results, h...",NovaBean Heritage,Challenger,0,1,0,0.00,0.000,0,,NaN,NaN,{}
4,Coffee Beans,Experience Good,1,1_Baseline,3,"Based solely on the provided search results, h...",Ethiopia Yirgacheffe Reserve,Leader,1,0,1,1.00,0.185,39,1. Ethiopia Yirgacheffe Reserve – This is the ...,NaN,NaN,NaN
